# 01 - Preprocessing

Runs the three pipeline steps from `src/` and shows each result inline. The logic lives in `src/` (one source of truth), so this notebook and `python run_pipeline.py all` always do exactly the same thing.

**Before running:** put the dataset CSV in `data/raw/` (it must be the only `.csv` there, or set `raw.file` in `config/config.yaml`).

Run the cells top to bottom.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

import pandas as pd

# Works whether Jupyter was started in the project root or in notebooks/
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
assert (ROOT / "src").exists(), f"Project root not found from {Path.cwd()}"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import clean, convert, profiling
from src import settings as S

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 80)
print("Project root:", ROOT)

## 1. Convert: CSV -> Parquet (lossless)

Reads the CSV in chunks and stores every column as text, so nothing is altered (leading zeros kept, `NA` not treated as missing). Output: `data/interim/transactions_raw.parquet`.

In [ ]:
convert.run()

Original header names and the normalised names used from here on:

In [ ]:
pd.read_csv(S.COLUMN_MAPPING_CSV, keep_default_na=False)

## 2. Profile every column

Scans the whole file. Check here: which columns are empty, which look numeric, which have leading zeros (identifiers), and the most frequent values.

In [ ]:
profiling.run()

In [ ]:
profile_summary = pd.read_csv(S.PROFILE_CSV, keep_default_na=False)
profile_summary

## 3. Clean: assign types and cast

Types are inferred from the full-file profile. The step refuses to save if the row count changes or a cast creates a missing value. Output: `data/processed/transactions_clean.parquet`.

In [ ]:
clean.run()

The type chosen for each column and why. **Review this table.** If a type is wrong, or a column should be dropped or renamed, edit `clean.force_types`, `clean.drop_columns` or `clean.rename` in `config/config.yaml`, then re-run the `clean.run()` cell and this one (no kernel restart needed).

In [ ]:
pd.read_csv(S.TYPE_PLAN_CSV, keep_default_na=False)

## 4. Load and check the clean data

This loads the whole typed file into memory. If it does not fit, pass `columns=[...]` to `pd.read_parquet` to load only what you need.

In [ ]:
df = pd.read_parquet(S.CLEAN_PARQUET)
print(f"{len(df):,} rows x {df.shape[1]} columns, {df.memory_usage(deep=True).sum() / 1e6:,.1f} MB in memory")
df.head(10)

In [ ]:
pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(4),
    "distinct": df.nunique(),
})

In [ ]:
df.describe(include="number").T

Label balance (only if the label column from `config.yaml` exists):

In [ ]:
label = S.load_config()["clean"]["label_column"]
if "is_anomaly" in df.columns:
    print(df["is_anomaly"].value_counts().rename({0: "normal", 1: "anomaly"}).to_string())
    print(f"\nAnomaly rate: {df['is_anomaly'].mean() * 100:.4f}%\n")
    display(df[label].value_counts(dropna=False).to_frame("rows"))
else:
    print("is_anomaly was not created - check clean.label_column in config/config.yaml")